In [1]:
from pathlib import Path
import os
import json
import time
import hashlib
import uuid
import math
import random
import requests
import pandas as pd

from dotenv import load_dotenv
from tqdm.auto import tqdm
from qdrant_client import QdrantClient, models

# แก้ให้ตรงกับ project root ของคุณ
PROJECT_ROOT = Path(r"D:\Project_Github\gold assisstant")

INPUT_FILE = (
    PROJECT_ROOT
    / "embedding"
    / "artifacts"
    / "chunking_real"
    / "structure_aware"
    / "chunks_structure_aware_real.jsonl"
)

WORK_DIR = PROJECT_ROOT / "vector_database"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint files
EMBEDDINGS_FILE = WORK_DIR / "embeddings_checkpoint.jsonl"
EMBEDDING_ERRORS_FILE = WORK_DIR / "embedding_errors.jsonl"
UPSERT_CHECKPOINT_FILE = WORK_DIR / "upsert_checkpoint.json"

# Load secrets from project .env
load_dotenv(PROJECT_ROOT / ".env")

CF_ACCOUNT_ID = os.getenv("CLOUDFLARE_ACCOUNT_ID")
CF_API_TOKEN = os.getenv("CLOUDFLARE_API_TOKEN")

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

MODEL_NAME = "@cf/baai/bge-m3"
COLLECTION_NAME = "gold_news_structure_aware"

# Tune these based on API limits and chunk length
EMBED_BATCH_SIZE = 16
UPSERT_BATCH_SIZE = 64
MAX_RETRIES = 6
REQUEST_TIMEOUT = 120

assert INPUT_FILE.exists(), f"Input file not found: {INPUT_FILE}"

missing = [
    name for name, value in {
        "CLOUDFLARE_ACCOUNT_ID": CF_ACCOUNT_ID,
        "CLOUDFLARE_API_TOKEN": CF_API_TOKEN,
        "QDRANT_URL": QDRANT_URL,
        "QDRANT_API_KEY": QDRANT_API_KEY,
    }.items()
    if not value
]

if missing:
    raise EnvironmentError(
        "Missing environment variables: " + ", ".join(missing)
    )

CF_ENDPOINT = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{CF_ACCOUNT_ID}/ai/run/{MODEL_NAME}"
)

print("Input:", INPUT_FILE)
print("Work directory:", WORK_DIR)
print("Embedding model:", MODEL_NAME)
print("Qdrant collection:", COLLECTION_NAME)
print("Secrets loaded:", True)

Input: D:\Project_Github\gold assisstant\embedding\artifacts\chunking_real\structure_aware\chunks_structure_aware_real.jsonl
Work directory: D:\Project_Github\gold assisstant\vector_database
Embedding model: @cf/baai/bge-m3
Qdrant collection: gold_news_structure_aware
Secrets loaded: True


In [2]:
def read_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSON at line {line_no}: {e}"
                ) from e

    return records


def extract_chunk_text(record):
    # แก้รายการนี้ได้ หาก schema จริงใช้ชื่อ field อื่น
    candidate_fields = (
        "text",
        "chunk_text",
        "content",
        "page_content",
        "body",
    )

    for field in candidate_fields:
        value = record.get(field)
        if isinstance(value, str) and value.strip():
            return value.strip()

    raise ValueError(
        "Cannot find chunk text. "
        f"Available keys: {list(record.keys())}"
    )


raw_records = read_jsonl(INPUT_FILE)

if not raw_records:
    raise ValueError("Input JSONL contains no records.")

print("Raw records:", len(raw_records))
print("Example keys:", list(raw_records[0].keys()))

# Preview the first record's schema without dumping the full corpus
print(json.dumps(raw_records[0], ensure_ascii=False, indent=2)[:2500])

Raw records: 19784
Example keys: ['chunk_id', 'schema_version', 'source', 'document_id', 'original_id', 'chunk_index', 'section_index', 'part_index', 'title', 'section_heading', 'publication_date', 'report_period', 'language', 'url', 'source_file', 'source_classification', 'content', 'char_count', 'approx_token_count']
{
  "chunk_id": "ce67d31dde131c6cb141",
  "schema_version": "structure_aware_real_v1",
  "source": "thai_pbs_gold",
  "document_id": "b69cb2366d22735f",
  "original_id": "thaipbs:news:571381",
  "chunk_index": 1,
  "section_index": 0,
  "part_index": 0,
  "title": "สรุปราคาทองคำ 20 ต.ค.2569 ปิดตลาด ผันผวน 21 ครั้ง ลบ 650 บาท",
  "section_heading": "Abstract / Body",
  "publication_date": "2026-10-20T17:52:29+07:00",
  "report_period": "",
  "language": "th",
  "url": "",
  "source_file": "D:\\Project_Github\\gold assisstant\\dataset\\news\\pbs\\gold_audit_direct.jsonl",
  "source_classification": {
    "relevance": "direct",
    "confidence": 90,
    "reason": "ข่าวรายงา

In [3]:
chunks_by_id = {}
duplicate_count = 0
empty_count = 0

for row_number, record in enumerate(raw_records, start=1):
    chunk_id = record.get("chunk_id")

    if chunk_id is None or not str(chunk_id).strip():
        raise ValueError(
            f"Missing chunk_id at record #{row_number}. "
            "Inspect the source schema before continuing."
        )

    chunk_id = str(chunk_id).strip()

    try:
        chunk_text = extract_chunk_text(record)
    except ValueError as e:
        raise ValueError(
            f"Cannot extract text for chunk_id={chunk_id!r}: {e}"
        ) from e

    if not chunk_text:
        empty_count += 1
        continue

    if chunk_id in chunks_by_id:
        duplicate_count += 1
        continue

    normalized = dict(record)
    normalized["chunk_id"] = chunk_id
    normalized["_embedding_text"] = chunk_text

    chunks_by_id[chunk_id] = normalized


chunks = list(chunks_by_id.values())

if not chunks:
    raise ValueError("No usable chunks found.")

print(f"Records read:       {len(raw_records):,}")
print(f"Unique chunk IDs:   {len(chunks):,}")
print(f"Duplicate IDs:      {duplicate_count:,}")
print(f"Empty text records: {empty_count:,}")
print("First chunk ID:", chunks[0]["chunk_id"])
print("First chunk text:", chunks[0]["_embedding_text"][:300])

Records read:       19,784
Unique chunk IDs:   19,784
Duplicate IDs:      0
Empty text records: 0
First chunk ID: ce67d31dde131c6cb141
First chunk text: ราคาทองคำ ลบ 650 บาท ผันผวน 21 ครั้ง ทองลงขณะน้ำมันดิบพุ่ง หลังสหรัฐฯ – อิสราเอล กำลังพิจารณาโจมตีอิหร่าน แต่ดอลลาร์กลับอ่อนค่า "รูปพรรณ" ขายออก 66,400 บาททองคำ "ทองคำแท่ง" ขายออก 65,600 บาททองคำ

วันนี้ (20 ต.ค.2569) "ฮั่วเซ่งเฮง" เปิดเผยภาพรวมราคาทองโลกที่ผ่านมา ราคาทองคำมีแนวโน้มย่อตัวลง จากราคาน


In [4]:
cf_session = requests.Session()
cf_session.headers.update({
    "Authorization": f"Bearer {CF_API_TOKEN}",
    "Content-Type": "application/json",
})

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

# Test Qdrant credentials/connectivity
print("Qdrant collections:", [
    c.name for c in qdrant.get_collections().collections
])

Qdrant collections: ['gold_news_structure_aware']


In [5]:
def post_with_retry(url, *, headers=None, json_body=None, timeout=REQUEST_TIMEOUT):
    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(
                url,
                headers=headers,
                json=json_body,
                timeout=timeout,
            )

            # Retry transient errors and rate limits
            if response.status_code == 429 or response.status_code >= 500:
                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    wait_seconds = min(float(retry_after), 90)
                else:
                    wait_seconds = min(2 ** attempt + random.random(), 60)

                if attempt == MAX_RETRIES - 1:
                    response.raise_for_status()

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            data = response.json()

            if data.get("success") is False:
                raise RuntimeError(
                    f"Cloudflare API returned success=false: "
                    f"{data.get('errors', [])}"
                )

            if "result" not in data:
                raise RuntimeError(
                    f"Unexpected Cloudflare response keys: {list(data.keys())}"
                )

            return data["result"]

        except (requests.Timeout, requests.ConnectionError) as e:
            last_error = e
            if attempt == MAX_RETRIES - 1:
                break
            time.sleep(min(2 ** attempt + random.random(), 60))

        except requests.HTTPError as e:
            # Do not retry ordinary 4xx errors such as invalid credentials.
            raise RuntimeError(
                f"Cloudflare HTTP error {e.response.status_code}: "
                f"{e.response.text[:1000]}"
            ) from e

    raise RuntimeError(f"Cloudflare request failed after retries: {last_error}")


def embed_texts(texts):
    if not texts:
        return []

    result = post_with_retry(
        CF_ENDPOINT,
        headers=cf_session.headers,
        json_body={"text": texts},
    )

    # Workers AI embedding responses commonly expose result.data.
    vectors = result.get("data")

    if not isinstance(vectors, list):
        raise RuntimeError(
            "Embedding response has no list at result.data. "
            f"Returned keys: {list(result.keys())}"
        )

    if len(vectors) != len(texts):
        raise RuntimeError(
            f"Expected {len(texts)} vectors, received {len(vectors)}."
        )

    dimension = None
    validated = []

    for i, vector in enumerate(vectors):
        if not isinstance(vector, list) or not vector:
            raise RuntimeError(f"Invalid vector at batch index {i}.")

        if not all(
            isinstance(x, (int, float)) and math.isfinite(float(x))
            for x in vector
        ):
            raise RuntimeError(f"Non-finite or non-numeric values in vector {i}.")

        if dimension is None:
            dimension = len(vector)
        elif len(vector) != dimension:
            raise RuntimeError("Inconsistent vector dimensions in one batch.")

        validated.append([float(x) for x in vector])

    return validated


# Test using one real chunk
test_vectors = embed_texts([chunks[0]["_embedding_text"]])
VECTOR_DIMENSION = len(test_vectors[0])

print("Model:", MODEL_NAME)
print("Detected vector dimension:", VECTOR_DIMENSION)
print("Test vector length:", len(test_vectors[0]))
print("First 8 values:", test_vectors[0][:8])

Model: @cf/baai/bge-m3
Detected vector dimension: 1024
Test vector length: 1024
First 8 values: [-0.01169586181640625, -0.00904083251953125, -0.0281982421875, 0.036712646484375, -0.01013946533203125, -0.047332763671875, 0.006320953369140625, 0.0469970703125]


In [6]:
def get_existing_vector_size(collection_info):
    vectors_config = collection_info.config.params.vectors

    # This notebook uses a single unnamed dense vector.
    if isinstance(vectors_config, dict):
        if len(vectors_config) != 1:
            raise RuntimeError(
                "Existing collection uses named/multiple vectors. "
                "Please inspect its vector configuration."
            )
        vectors_config = next(iter(vectors_config.values()))

    size = getattr(vectors_config, "size", None)

    if size is None:
        raise RuntimeError(
            "Could not determine existing collection vector size."
        )

    return int(size)


existing_names = {
    c.name for c in qdrant.get_collections().collections
}

if COLLECTION_NAME not in existing_names:
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=VECTOR_DIMENSION,
            distance=models.Distance.COSINE,
        ),
    )
    print("Created collection:", COLLECTION_NAME)

else:
    info = qdrant.get_collection(COLLECTION_NAME)
    existing_dimension = get_existing_vector_size(info)

    if existing_dimension != VECTOR_DIMENSION:
        raise ValueError(
            f"Collection dimension mismatch: existing={existing_dimension}, "
            f"new={VECTOR_DIMENSION}. "
            "Use a new collection name or migrate intentionally; "
            "this notebook will not delete existing data."
        )

    print("Using existing collection:", COLLECTION_NAME)
    print("Existing vector dimension:", existing_dimension)

print("Collection points before ingestion:",
      qdrant.count(COLLECTION_NAME, exact=True).count)

Using existing collection: gold_news_structure_aware
Existing vector dimension: 1024
Collection points before ingestion: 19784


In [7]:
def text_sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_checkpoint_jsonl(path):
    saved = {}

    if not path.exists():
        return saved

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                print(f"Warning: skipping invalid checkpoint line {line_no}")
                continue

            chunk_id = row.get("chunk_id")
            if chunk_id:
                # Latest valid row wins if a chunk was re-embedded.
                saved[str(chunk_id)] = row

    return saved


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


saved_embeddings = read_checkpoint_jsonl(EMBEDDINGS_FILE)

pending_chunks = []

for chunk in chunks:
    chunk_id = chunk["chunk_id"]
    text = chunk["_embedding_text"]
    expected_hash = text_sha256(text)

    cached = saved_embeddings.get(chunk_id)

    cache_is_valid = (
        cached is not None
        and cached.get("text_sha256") == expected_hash
        and cached.get("model") == MODEL_NAME
        and cached.get("dimension") == VECTOR_DIMENSION
        and isinstance(cached.get("vector"), list)
        and len(cached["vector"]) == VECTOR_DIMENSION
    )

    if not cache_is_valid:
        pending_chunks.append(chunk)

print("Total unique chunks:", len(chunks))
print("Valid cached embeddings:", len(chunks) - len(pending_chunks))
print("Chunks still to embed:", len(pending_chunks))


def run_embedding_batches(pending):
    completed = 0
    failed = []

    for start in tqdm(range(0, len(pending), EMBED_BATCH_SIZE)):
        batch = pending[start:start + EMBED_BATCH_SIZE]
        texts = [row["_embedding_text"] for row in batch]

        try:
            vectors = embed_texts(texts)

            for row, vector in zip(batch, vectors):
                checkpoint_record = {
                    "chunk_id": row["chunk_id"],
                    "text_sha256": text_sha256(row["_embedding_text"]),
                    "model": MODEL_NAME,
                    "dimension": len(vector),
                    "vector": vector,
                }
                append_jsonl(EMBEDDINGS_FILE, checkpoint_record)
                saved_embeddings[row["chunk_id"]] = checkpoint_record

            completed += len(batch)

        except Exception as e:
            error_record = {
                "time": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "chunk_ids": [row["chunk_id"] for row in batch],
                "error": repr(e),
            }
            append_jsonl(EMBEDDING_ERRORS_FILE, error_record)
            failed.extend(row["chunk_id"] for row in batch)

            print(
                f"\nBatch failed at offset {start}. "
                f"Error logged; rerun this cell after resolving the issue.\n{e}"
            )

            # Stop rather than silently skipping failed chunks.
            break

    return completed, failed


embedded_now, failed_ids = run_embedding_batches(pending_chunks)

print("New embeddings saved:", embedded_now)
print("Failed chunk IDs:", len(failed_ids))
print("Checkpoint file:", EMBEDDINGS_FILE)

if failed_ids:
    print("Some chunks remain unembedded. Resolve the error and rerun Cell 8.")
else:
    print("Embedding batches completed.")

Total unique chunks: 19784
Valid cached embeddings: 19784
Chunks still to embed: 0


0it [00:00, ?it/s]

New embeddings saved: 0
Failed chunk IDs: 0
Checkpoint file: D:\Project_Github\gold assisstant\vector_database\embeddings_checkpoint.jsonl
Embedding batches completed.


In [8]:
def qdrant_point_id(chunk_id):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"gold-news:{chunk_id}"))


def json_safe(value):
    """Convert common Python values to JSON-compatible metadata."""
    if value is None or isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, float):
        return value if math.isfinite(value) else None

    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    return str(value)


def read_upsert_checkpoint(path):
    saved = set()

    # Recover both the last committed snapshot and a complete temp snapshot
    # left behind when Windows refuses to replace a file held open elsewhere.
    for json_path in (path, path.with_suffix(".tmp")):
        if not json_path.exists():
            continue
        try:
            data = json.loads(json_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as exc:
            print(f"Warning: could not read checkpoint {json_path}: {exc}")
            continue
        if isinstance(data, dict):
            saved.update(str(value) for value in data.get("upserted_chunk_ids", []) if value)

    # Append-only journal avoids replacing a locked checkpoint on Windows.
    journal_path = path.with_suffix(".jsonl")
    if journal_path.exists():
        with journal_path.open("r", encoding="utf-8") as file:
            for line_no, line in enumerate(file, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError as exc:
                    print(f"Warning: skipping incomplete checkpoint journal line {line_no}: {exc}")
                    continue
                saved.update(
                    str(value) for value in record.get("upserted_chunk_ids", []) if value
                )

    return saved


persisted_upsert_ids = read_upsert_checkpoint(UPSERT_CHECKPOINT_FILE)
upserted_ids = set(persisted_upsert_ids)


def write_upsert_checkpoint(path, upserted_ids):
    new_ids = sorted(set(upserted_ids) - persisted_upsert_ids)
    if not new_ids:
        return

    journal_path = path.with_suffix(".jsonl")
    with journal_path.open("a", encoding="utf-8", newline="\n") as file:
        file.write(json.dumps({"upserted_chunk_ids": new_ids}, ensure_ascii=False) + "\n")
        file.flush()
        os.fsync(file.fileno())
    persisted_upsert_ids.update(new_ids)

# Map chunk IDs to the latest input records
chunk_lookup = {row["chunk_id"]: row for row in chunks}

# Only upload embeddings that match current text
upload_rows = []

for chunk_id, saved in saved_embeddings.items():
    chunk = chunk_lookup.get(chunk_id)
    if chunk is None:
        continue

    if saved.get("text_sha256") != text_sha256(chunk["_embedding_text"]):
        continue

    if saved.get("model") != MODEL_NAME:
        continue

    if len(saved.get("vector", [])) != VECTOR_DIMENSION:
        continue

    if chunk_id not in upserted_ids:
        upload_rows.append((chunk, saved["vector"]))


def make_qdrant_point(chunk, vector):
    payload = {
        # Keep retrieval text and provenance
        **{
            k: json_safe(v)
            for k, v in chunk.items()
            if not k.startswith("_") and k != "vector"
        },
        "chunk_id": chunk["chunk_id"],
        "text": chunk["_embedding_text"],
        "embedding_model": MODEL_NAME,
        "embedding_dimension": VECTOR_DIMENSION,
        "text_sha256": text_sha256(chunk["_embedding_text"]),
    }

    return models.PointStruct(
        id=qdrant_point_id(chunk["chunk_id"]),
        vector=vector,
        payload=payload,
    )


def read_uploaded_chunk_ids_from_qdrant():
    """Find already-uploaded points so a lost checkpoint never causes needless rework."""
    found = set()
    offset = None
    while True:
        points, offset = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            limit=512,
            offset=offset,
            with_payload=["chunk_id", "text_sha256"],
            with_vectors=False,
        )
        for point in points:
            payload = point.payload or {}
            chunk_id = str(payload.get("chunk_id") or "")
            chunk = chunk_lookup.get(chunk_id)
            if chunk is None or point.id != qdrant_point_id(chunk_id):
                continue
            if payload.get("text_sha256") == text_sha256(chunk["_embedding_text"]):
                found.add(chunk_id)
        if offset is None:
            return found


remote_confirmed_ids = read_uploaded_chunk_ids_from_qdrant()
new_remote_ids = remote_confirmed_ids - upserted_ids
if new_remote_ids:
    upserted_ids.update(new_remote_ids)
    write_upsert_checkpoint(UPSERT_CHECKPOINT_FILE, upserted_ids)

upload_rows = [
    (chunk, vector)
    for chunk, vector in upload_rows
    if chunk["chunk_id"] not in upserted_ids
]

print("Cached embeddings eligible for upload:", len(upload_rows))
print("Already confirmed by Qdrant/checkpoint:", len(upserted_ids & set(chunk_lookup)))
print("Remaining chunks to upload:", len(upload_rows))


def run_qdrant_upsert(rows):
    for start in tqdm(range(0, len(rows), UPSERT_BATCH_SIZE)):
        batch = rows[start:start + UPSERT_BATCH_SIZE]
        points = [
            make_qdrant_point(chunk, vector)
            for chunk, vector in batch
        ]

        # Retry the same deterministic point IDs on transient errors.
        last_error = None

        for attempt in range(MAX_RETRIES):
            try:
                qdrant.upsert(
                    collection_name=COLLECTION_NAME,
                    points=points,
                    wait=True,
                )
                last_error = None
                break

            except Exception as e:
                last_error = e

                if attempt == MAX_RETRIES - 1:
                    break

                time.sleep(min(2 ** attempt + random.random(), 60))

        if last_error is not None:
            raise RuntimeError(
                f"Qdrant upsert failed at offset {start}; "
                "rerun Cell 9 after resolving the issue. "
                "Repeated upsert is safe because point IDs are deterministic."
            ) from last_error

        # Persist only after Qdrant confirms the batch.
        upserted_ids.update(chunk["chunk_id"] for chunk, _ in batch)

        write_upsert_checkpoint(UPSERT_CHECKPOINT_FILE, upserted_ids)


run_qdrant_upsert(upload_rows)

print("Qdrant upsert journal:", UPSERT_CHECKPOINT_FILE.with_suffix(".jsonl"))
print("Chunks marked uploaded:", len(upserted_ids & set(chunk_lookup)))
final_point_count = qdrant.count(COLLECTION_NAME, exact=True).count
print("Qdrant exact point count:", final_point_count)
if not set(chunk_lookup).issubset(upserted_ids):
    raise RuntimeError("Some input chunks are still not confirmed uploaded; rerun this cell to resume.")
print("All input chunks are confirmed uploaded.")

Cached embeddings eligible for upload: 0
Already confirmed by Qdrant/checkpoint: 19784
Remaining chunks to upload: 0


0it [00:00, ?it/s]

Qdrant upsert journal: D:\Project_Github\gold assisstant\vector_database\upsert_checkpoint.jsonl
Chunks marked uploaded: 19784
Qdrant exact point count: 19784
All input chunks are confirmed uploaded.


In [9]:
def search_gold_news(query_text, top_k=5):
    query_vector = embed_texts([query_text])[0]

    if len(query_vector) != VECTOR_DIMENSION:
        raise ValueError(
            f"Query vector dimension {len(query_vector)} "
            f"does not match collection dimension {VECTOR_DIMENSION}."
        )

    result = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )

    output = []

    for rank, point in enumerate(result.points, start=1):
        payload = point.payload or {}

        output.append({
            "rank": rank,
            "score": point.score,
            "chunk_id": payload.get("chunk_id"),
            "title": (
                payload.get("title")
                or payload.get("headline")
                or payload.get("document_title")
            ),
            "source": (
                payload.get("source")
                or payload.get("source_name")
                or payload.get("url")
            ),
            "text": payload.get("text", "")[:1000],
        })

    return pd.DataFrame(output)


test_query = (
    "How could higher-than-expected US inflation affect gold prices "
    "through interest rate expectations and the US dollar?"
)

search_results = search_gold_news(test_query, top_k=5)
display(search_results)

,rank,score,chunk_id,title,source,text
0,1,0.612063,69163f25f99c2d765db5,"ราคา ""ทองคำ"" เช้านี้ -100 บาท ""รูปพรรณ"" ขายออก...",thai_pbs_gold,"ราคา ""ทองคำ"" เช้านี้เปิดตลาด -100 บาท (ครั้งที..."
1,2,0.599427,37de06bdb8a88055745b,สรุปราคาทองคำ 20 ต.ค.2569 ปิดตลาด ผันผวน 21 คร...,thai_pbs_gold,อย่างไรก็ตาม ตลาดยังคงคาดการณ์เกิน 80% ว่าเฟดอ...
2,3,0.583873,ccc29832b5b4d2200d80,สรุปราคาทองคำ 21ต.ค.2569 ผันผวน 20 ครั้ง ไม่เป...,thai_pbs_gold,ในขณะที่ราคาน้ำมันดิบมีแนวโน้มปรับตัวขึ้นต่อ ห...
3,4,0.581874,59a5b15bb4e7097a5dfd,สรุปราคาทองคำ 14 ต.ค.2569 ปิดตลาด ผันผวน 19 คร...,thai_pbs_gold,ตัวเลขเศรษฐกิจที่ต้องติดตามคืนนี้ เวลา 19.30 น...
4,5,0.581675,20b55103087500649fc7,"ราคา ""ทองคำ"" เช้านี้ -100 บาท ""รูปพรรณ"" ขายออก...",thai_pbs_gold,ประเด็นที่ต้องติดตามคืนนี้ เวลา 19.30 น. สหรัฐ...


In [10]:
thai_query = (
    "เงินเฟ้อสหรัฐสูงกว่าคาด ส่งผลต่อการคาดการณ์ดอกเบี้ย "
    "ค่าเงินดอลลาร์ และราคาทองคำอย่างไร"
)

display(search_gold_news(thai_query, top_k=5))

,rank,score,chunk_id,title,source,text
0,1,0.680493,69163f25f99c2d765db5,"ราคา ""ทองคำ"" เช้านี้ -100 บาท ""รูปพรรณ"" ขายออก...",thai_pbs_gold,"ราคา ""ทองคำ"" เช้านี้เปิดตลาด -100 บาท (ครั้งที..."
1,2,0.647978,37de06bdb8a88055745b,สรุปราคาทองคำ 20 ต.ค.2569 ปิดตลาด ผันผวน 21 คร...,thai_pbs_gold,อย่างไรก็ตาม ตลาดยังคงคาดการณ์เกิน 80% ว่าเฟดอ...
2,3,0.641596,56c2f68bfd6537c01ef3,ราคา“ทองคำ” เช้านี้ -300 บาท “รูปพรรณ” ขายออก ...,thai_pbs_gold,ขณะที่ทางด้านสหรัฐฯ จอห์น วิลเลียมส์ ประธานธนา...
3,4,0.623650,ccc29832b5b4d2200d80,สรุปราคาทองคำ 21ต.ค.2569 ผันผวน 20 ครั้ง ไม่เป...,thai_pbs_gold,ในขณะที่ราคาน้ำมันดิบมีแนวโน้มปรับตัวขึ้นต่อ ห...
4,5,0.617150,f65fe85fd838b532d8d7,"ราคา ""ทองคำ"" เช้านี้ -100 บาท ""รูปพรรณ"" ขายออก...",thai_pbs_gold,โดย FedWatch ล่าสุดให้น้ำหนักมากกว่า 50% จากเพ...


In [11]:
current_ids = {row["chunk_id"] for row in chunks}

embedded_valid_ids = {
    chunk_id
    for chunk_id, saved in saved_embeddings.items()
    if (
        chunk_id in current_ids
        and saved.get("text_sha256")
            == text_sha256(chunk_lookup[chunk_id]["_embedding_text"])
        and saved.get("model") == MODEL_NAME
        and len(saved.get("vector", [])) == VECTOR_DIMENSION
    )
}

missing_embeddings = current_ids - embedded_valid_ids

print("Unique input chunks:       ", len(current_ids))
print("Valid embeddings:          ", len(embedded_valid_ids))
print("Missing embeddings:        ", len(missing_embeddings))
print("Qdrant points (exact):      ",
      qdrant.count(COLLECTION_NAME, exact=True).count)
print("Detected vector dimension: ", VECTOR_DIMENSION)

if missing_embeddings:
    print("Example missing IDs:", list(missing_embeddings)[:10])
else:
    print("All current chunks have valid cached embeddings.")

Unique input chunks:        19784
Valid embeddings:           19784
Missing embeddings:         0
Qdrant points (exact):       19784
Detected vector dimension:  1024
All current chunks have valid cached embeddings.
